In [6]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY)

In [10]:
import bs4
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [9]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [13]:
loader = WebBaseLoader(
  web_paths=("https://docs.langchain.com/oss/python/integrations/embeddings",),
)

docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/integrations/embeddings', 'title': 'Embedding model integrations - Docs by LangChain', 'description': 'Integrate with embedding models using LangChain Python.', 'language': 'en'}, page_content='Embedding model integrations - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationIntegrations by componentEmbedding model integrationsOverviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonLangChain integrationsAll providersPopular ProvidersOpenAIAnthropicGoogleAWSNVIDIAHugging FaceMicrosoftOllamaGroqFireworksIntegrations by componentChat modelsTools and toolkitsMiddlewareSandboxesCheckpointersRetrieversText splittersEmbedding modelsVector storesDocument loadersOn this pageOvervi

In [14]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

splits = splitter.split_documents(docs)

# Storing the Splitted docs in ChromaDB
vector_store = Chroma.from_documents(documents=splits, embedding=embedding)

retriever = vector_store.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002A5B9CFD550>, search_kwargs={})

In [15]:
# Prompt Template
system_prompt = (
  "You are an assistant for question-answering tasks. Use the following piece of retrieved context to answer the question. If you don't know the answer, then say that you don't know it. Use five or less sentences the maximum and keep the answers concise \n\n {context}"
)

prompt = ChatPromptTemplate.from_messages(
  [
    ("system", system_prompt),
    ("user", "{input}")
  ]
)

In [17]:
question_answer_chain = create_stuff_documents_chain(model, prompt=prompt)

rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [18]:
response = rag_chain.invoke({"input": "What is similarity metrics"})
response

{'input': 'What is similarity metrics',
 'context': [Document(id='804c1d1f-0fbe-4db2-83d5-9d14cdccb430', metadata={'description': 'Integrate with embedding models using LangChain Python.', 'title': 'Embedding model integrations - Docs by LangChain', 'source': 'https://docs.langchain.com/oss/python/integrations/embeddings', 'language': 'en'}, page_content='Vectorization — The model encodes each input string as a high-dimensional vector.\nSimilarity scoring — Vectors are compared using mathematical metrics to measure how closely related the underlying texts are.\n\n\u200bSimilarity metrics\nSeveral metrics are commonly used to compare embeddings:\n\nCosine similarity — measures the angle between two vectors.\nEuclidean distance — measures the straight-line distance between points.\nDot product — measures how much one vector projects onto another.\n\nHere’s an example of computing cosine similarity between two vectors:\nimport numpy as np\n\ndef cosine_similarity(vec1, vec2):\n    dot = n

In [19]:
response['answer']

'Similarity metrics are used to compare vectors (embeddings) and measure how closely related the underlying texts are. Commonly used metrics include:\n\n1. Cosine similarity\n2. Euclidean distance\n3. Dot product'

In [22]:
rag_chain.invoke({"input": "What are its various types"})['answer']

# This answwr generated is no where related to similarity metrics and hence we need a chat history or conversational history

'The Embeddings interface in LangChain provides a standard interface for text embedding models. It offers two main methods: embed_documents and embed_query.'

In [23]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

contextualize_question_system_prompt = (
  "Given a chat history and the latest user question which might reference context in the chat history, formulate a standalone question which can be understood without the chat history. Do not answer the question, just reformulate it if needed and otherwise return it as it is."
)

contextualize_question_prompt = ChatPromptTemplate.from_messages(
  [
    ("system", contextualize_question_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("user", "{input}")
  ]
)

In [24]:
history_aware_retriever = create_history_aware_retriever(model, retriever, contextualize_question_prompt)

history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000002A5B9CFD550>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessag

In [29]:
qa_prompt = ChatPromptTemplate.from_messages(
  [
    (
      "system",
      "You are a question-answering assistant. Use the retrieved context to answer the user's question. "
      "If the answer is not contained in the context, say that you don't know.\n\n{context}"
    ),
    MessagesPlaceholder("chat_history"),
    ("user", "{input}")
  ]
)

In [30]:
question_answer_chain = create_stuff_documents_chain(llm=model, prompt=qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [32]:
from langchain_core.messages import AIMessage, HumanMessage

history = []

question1 = "What is similarity score?"
res1 = rag_chain.invoke({"input": question1, "chat_history": history})

history.extend(
  [
    HumanMessage(content=question1),
    AIMessage(content=res1['answer'])
  ]
)

question2 = "What are its different types?"
res2 = rag_chain.invoke({"input": question2, "chat_history": history})

print(res1["answer"])
print(res2["answer"])

history.extend(
  [
    HumanMessage(content=question2),
    AIMessage(content=res2['answer'])
  ]
)

A similarity score is a value that measures how closely related two pieces of text are. It is calculated by comparing the vector representations of the two texts, which are generated by text embedding models. The similarity score is often used to determine the relevance or similarity between a query and a document, or between two documents.

In the context of the provided text, similarity scoring is achieved by comparing the vectors generated by text embedding models using mathematical metrics, such as cosine similarity, Euclidean distance, or dot product. The similarity score is then calculated based on these metrics, with higher scores indicating a closer relationship between the two texts.
According to the provided text, there are several types of similarity metrics used to compare embeddings:

1. Cosine similarity: measures the angle between two vectors.
2. Euclidean distance: measures the straight-line distance between points.
3. Dot product: measures how much one vector projects 